In [ ]:
## Load modeules
import torch
import os
from tqdm import trange
import py3Dmol

from sinatra_pro.atomic import convert_trj_to_meshes
from sinatra_pro.mesh import Mesh
from sinatra_pro.directions import generate_equidistributed_cones
from sinatra_pro.euler import compute_ec_curve
from sinatra_pro.rate import calc_rate
from sinatra_pro.reconstruction import reconstruct_by_sorted_threshold, write_vert_prob_on_pdb, write_vert_prob_on_pdb_residue

In [ ]:
## Plotting library and settings
import matplotlib as mpl
from matplotlib import pyplot as plt

mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['xtick.labelsize'] = 16
mpl.rcParams['ytick.labelsize'] = 16
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['lines.linewidth'] = 4
mpl.rcParams['lines.markersize'] = 6
mpl.rcParams['axes.linewidth'] = 4
mpl.rcParams['xtick.major.width'] = 4
mpl.rcParams['ytick.major.width'] = 4
mpl.rcParams['xtick.major.size'] = 10
mpl.rcParams['ytick.major.size'] = 10
mpl.rcParams['figure.dpi'] = 100
mpl.rcParams['legend.facecolor'] = 'white'
mpl.rcParams['legend.edgecolor'] = 'white'
mpl.rcParams['legend.frameon'] = True

PLOT_ALPHA = 0.8

colors = [
    "#000000", "#E69F00", "#56B4E9",
    "#009E73", "#F0E442", "#0072B2",
    "#D55E00", "#CC79A7",
]
linestyles = [
    "-", "--", "-.", ":"
]


In [ ]:
## Input for the analysis
dir_out = "./output/" ### output directory
trj_prot_A = "./data/WT_10.xtc"
top_prot_A =  "./data/WT.gro"
trj_prot_B = "./data/R164S_10.xtc"
top_prot_B =  "./data/R164S.gro"

## These are recommended settings that works generally
## TODO: automatically adapt parameters to data
align_selection = "protein and name CA"
mesh_selection = "protein and not type H"
align_sequence = True
radius_sim = 3.0 # radius for simplicial construction
hemisphere = True # distribute directions over hemisphere instead of whole sphere
ec_type = 'DECT' # type of Euler characteristic measure (DECT/ECT/SECT), default: DECT', default='DECT'
n_cones = 20 # number of cone
n_direction_per_cone = 3 # number of direction per cone
cap_radius = 0.8 # cap radius for each cone
n_filtrations = 60 # number of filtration step
bandwidth = 0.01 # bandwidth for elliptical slice sampling
n_mcmc_steps = 2000 # number of sample from ESS
n_burn_in_steps = 1000 # number of burn in steps for MCMC
probit = False # use logistic likelihood instead of probit likelihood
low_rank = True # use low rank matrix approximations to compute the RATE values
verbose = True
os.makedirs(dir_out, exist_ok=True)

In [ ]:
### Read PDB files for coordinates and Convert PDB to meshes
print("Converting PDB to meshes...")
meshes, labels = convert_trj_to_meshes(
    trj_file_A=trj_prot_A,
    top_file_A=top_prot_A,
    trj_file_B=trj_prot_B,
    top_file_B=top_prot_B,
    align_selection = align_selection,
    mesh_selection = mesh_selection,
    radius_sim = radius_sim,
    align_sequence = align_sequence,
    verbose = verbose
)
n_meshes = len(meshes)
labels = torch.tensor(labels, dtype=torch.float32)

In [ ]:
## Calculate distributed cones of directions for Euler Characteristics (EC) calculations
print("Generating directions for EC calculations")
directions = generate_equidistributed_cones(
    n_cones=n_cones,
    n_direction_per_cone=n_direction_per_cone,
    cap_radius=cap_radius,
    hemisphere=hemisphere
)
n_directions = directions.shape[0]
n_features = n_directions * n_filtrations

## Perform Euler Characteristics (EC) calculations to convert the topology into summary statistics
ec_curves_meshes = torch.zeros((n_meshes, n_features), dtype=torch.float32)
for i in trange(len(meshes), desc="Calculating EC for Meshes"):
    filtration_radius, ec_curves = compute_ec_curve(
        meshes[i], directions, n_filtrations, ball_radius = 1.0, ec_type = ec_type, include_faces = True
    )
    ec_curves_meshes[i, :] = torch.from_numpy(ec_curves.flatten())

ec_curves_meshes_transposed = ec_curves_meshes.T # (n_features, n_samples)
good_features = (ec_curves_meshes_transposed != 0.0).to(dtype=torch.float32).sum(dim=1) > 0

ec_curves_meshes_transposed_masked = ec_curves_meshes_transposed[good_features, :]
ec_curves_meshes_transposed_masked -= ec_curves_meshes_transposed_masked.mean(dim=1, keepdim=True)
ec_curves_meshes_transposed_masked /= ec_curves_meshes_transposed_masked.std(dim=1, keepdim=True)

## Plot EC of an example direction
for i in range(n_meshes):
    plt.plot(ec_curves_meshes[i,n_filtrations*10:n_filtrations*11])
plt.xlabel("z-position in one of the direction")
plt.ylabel("Topological summary statistics")

In [ ]:
## RATE calculation for variable selections from the topological summary statistics
result_rate = calc_rate(
    X = ec_curves_meshes_transposed_masked,
    y = labels,
    bandwidth = bandwidth,
    n_mcmc_steps = n_mcmc_steps,
    n_burn_in_steps = n_burn_in_steps,
    probit = probit,
    low_rank = low_rank,
    verbose = verbose
)
kld = result_rate['KLD']
rates = result_rate['RATE']
delta = result_rate['Delta']
eff_samp_size = result_rate['ESS']

In [ ]:
rates_unmasked = torch.zeros((n_features), dtype=torch.float32)
rates_unmasked[good_features] = rates

## reconstruct the RATE values onto the protein structures for visualization
## reconstruct probabilities are stored in "Temperature factor" column in the pdb format
## can then be visualized using py3dmol, Chimera or Pymol
vert_prob = reconstruct_by_sorted_threshold(
    meshes[0], 
    directions,
    rates_unmasked,
    n_filtrations, 
    n_direction_per_cone,
)
print(f"vert_prob.shape: {vert_prob.shape}")

In [ ]:
## Visualize results on reference structure
from pathlib import Path 
pdb_files_A_map = "./data/pdb/WT/WT_frame_0.pdb"
pdb_out_file = os.path.join(dir_out, "WT_frame_0_reco.pdb")
write_vert_prob_on_pdb(
    vert_prob = vert_prob, 
    pdb_in_file = pdb_files_A_map,
    pdb_out_file = pdb_out_file,
    selection = mesh_selection,
    by_rank = True
)
with open(pdb_out_file) as ifile:
    system = "".join([x for x in ifile])
view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(system)
view.setStyle({'model': -1}, {'cartoon':{'colorscheme':{'prop':'b','gradient':'sinebow','min':0,'max':100}}})
view.zoomTo()
view.show()